# Criação dos embeddings

Este notebook tem como objetivo principal extrair os alarmes únicos da base em sequências e utilizar esses textos para o treinamento de Word2Vecs.

A ideia é que ao mapear os dados dessa forma, alarmes que acontecem em situações parecidas carreguem informações parecidas, sendo assim, ao calcularmos a média com decaimento exponencial EWMA, teremos um vetor latente que contém o estado atual da máquina com base nos últimos erros apresentandos por ela.

Essas coordenadas serão utilizads como dados usados no processo de Machine Learning.

---

# Import dos dados e bibliotecas

In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 78.0 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np

from gensim.models import Word2Vec

In [3]:
df_caminhao = pd.read_parquet('/content/df_caminhao_limpo_filtrado.parquet', engine='pyarrow')

# Extração das sequências de erros

In [4]:
# Criando os limites de tempo para quebra de sequência
LIMITE_HORAS = 24
limite_tempo = pd.Timedelta(hours=LIMITE_HORAS)

# Garantindo que os eventos estejam ordenados para que o Word2Vec interprete corretamente
df_caminhao = df_caminhao.sort_values(by=['TAG', 'Data_Evento']).reset_index(drop=True)

# Calcula a diferença de tempo e cria um gatilho para facilitar a separação (o primeiro também recebe NaN)
df_caminhao['Diferenca_Tempo'] = df_caminhao.groupby('TAG')['Data_Evento'].diff()
df_caminhao['Nova_Sessao_Trigger'] = (df_caminhao['Diferenca_Tempo'] > limite_tempo) | df_caminhao['Diferenca_Tempo'].isna()

# Com base nos gatilhos, cria id para "sessões" que facilitam ainda mais a agregação
df_caminhao['ID_Sessao'] = df_caminhao.groupby('TAG')['Nova_Sessao_Trigger'].cumsum()

# Cria de fato as sequências
sequencias_finais = (
    df_caminhao
    .groupby(['TAG', 'ID_Sessao'])['Alarme']
    .apply(list)
    .tolist()
)

# Remove sequência muito curtas para o modelo contextual
sequencias_filtradas = [seq for seq in sequencias_finais if len(seq) >= 2]

In [5]:
sequencias_filtradas[0]

['mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'payload_overload_active',
 'left_rear_brake_temperature_active',
 'right_rear_brake_temperature_active',
 'payload_overload_active',
 'payload_overload_dump_active',
 'op_baixo_nivel_de_combustivel_20',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'payload_overload_active',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'payload_overload_active',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'mc

In [6]:
# Treina o modelo de Word2Vec

model_w2v = Word2Vec(
    sentences=sequencias_filtradas,
    vector_size=16,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=25,
    sample=1e-4
)

In [7]:
# Bloco de código que possibilita avaliar os alarmes que foram agrupados com base nas sequências

alarme_teste = "mc_temperatura_da_direcao_95c"

if alarme_teste in model_w2v.wv:
    print(f"Alarmes mais correlacionados com '{alarme_teste}':")
    for alarme, similaridade in model_w2v.wv.most_similar(alarme_teste, topn=15):
        print(f" - {alarme}: {similaridade:.4f}")

Alarmes mais correlacionados com 'mc_temperatura_da_direcao_95c':
 - torque_converter_screen_active: 0.8638
 - rear_aftercooler_temperature_active: 0.7684
 - steering_oil_temperature_active: 0.7666
 - engine_cool_temperature_active: 0.7563
 - engine_coolant_temperature_active: 0.6798
 - high_aftercooler_temperature_active: 0.6752
 - 3rd_tire_pressure_very_low: 0.6569
 - arc_automatic_retarder_control_ecm_active: 0.6422
 - right_front_brake_temperature_active: 0.6394
 - mc_temperatura_da_direc_o_95c: 0.6352
 - engine_coolant_fan_speed_active: 0.6130
 - boost_pressure_active: 0.5936
 - transmission_oil_level_active: 0.5702
 - ma_diferenca_de_temperatura_de_freio_te_td_25_c: 0.5613
 - unfiltered_oil_pressure_active: 0.5350


In [8]:
# Lista de erros que estão contidos na base de dados

model_w2v.wv.index_to_key

['final_drive_bypass_solenoid_active',
 '6th_tire_tag_timeout',
 '5th_tire_tag_timeout',
 'engine_coolant_level_active',
 '4th_tire_tag_timeout',
 '3rd_tire_tag_timeout',
 '2nd_tire_tag_timeout',
 '1st_tire_tag_timeout',
 'mc_temperatura_da_direcao_95c',
 'payload_overload_active',
 'steering_oil_temperature_active',
 'ma_baixa_voltagem_com_equip_desligado_24v',
 'shutdown_inputs_active',
 'differential_lube_pressure_active',
 'transmission_output_speed2_active',
 'start_motor_relay_active',
 'ma_dif_temp_exaust_60_oc',
 'engine_prelube_active',
 'brake_servomechanism_solenoid_active',
 'ambient_air_temperature_active',
 'mc_queda_pressao_do_oleo_motor_280kpa',
 'payload_overload_dump_active',
 'ma_diferenca_de_temperatura_de_freio_te_td_25_c',
 'tcs_falha_sensor_de_rotacao_da_roda_tras_esq',
 'mc_baixa_pressao_de_ar_de_freio_90psi',
 'differential_filter_switch_active',
 'mi_filtro_oleo_motor_obstruido_69kpa',
 'air_filter_active',
 'mc_queda_pressao_do_oleo_motor',
 'right_front_brak

# Extração de Embeddings e construção da média EWMA

In [9]:
# Bloco de código responśvel por obter as coordendas de embedding para cada um dos alarmes presente na base

tamanho_vetor = model_w2v.vector_size
colunas_vetor = [f'Dim_{i}' for i in range(tamanho_vetor)]

def obter_vetor(alarme):
    if alarme in model_w2v.wv:
        return model_w2v.wv[alarme]
    else:
        return np.zeros(tamanho_vetor)

vetores = df_caminhao['Alarme'].apply(obter_vetor)

# Retorna os embedding e concatena ao dados
df_vetores = pd.DataFrame(vetores.tolist(), index=df_caminhao.index, columns=colunas_vetor)
df_caminhao_vetorizado = pd.concat([df_caminhao, df_vetores], axis=1)

In [10]:
# Bloco de código responsável por calcular as médias com decaimento

# Importante que os dados estejam ordenados da forma certa e criamos um DataFrame auxiliar com a coluna de data como índice
df_enriquecido = df_caminhao_vetorizado.sort_values(by=['TAG', 'Data_Evento']).reset_index(drop=True)
df_calculo = df_enriquecido.set_index(pd.to_datetime(df_enriquecido['Data_Evento']))

# Cálcula da EWMA
meias_vidas = {'2h': '2 hours', '6h': '6 hours', '24h': '24 hours', '72h': '72 hours'}

for nome, tempo in meias_vidas.items():

    resultado_ewm = df_calculo.groupby('TAG')[colunas_vetor].ewm(halflife=tempo, times=df_calculo.index).mean()

    novas_colunas = [f'EWMA_{nome}_{col}' for col in colunas_vetor]
    df_enriquecido[novas_colunas] = resultado_ewm.values

df_enriquecido.drop(columns=colunas_vetor, inplace=True)

In [11]:
# Salva os arquivos com os embeddings

df_enriquecido.to_parquet('datasets/datasets_tratados/df_caminhao_modelagem_embeddings.parquet', index=False)